# ABI Wound Care — One-Click Train (A100)

**Runtime → A100 GPU**, then **Run this single cell**. No uploads.


In [ ]:
# One-click train — no uploads needed
!pip install -q pandas scikit-learn joblib

from pathlib import Path

FEATURES_CSV = """patient_internal_id,patient_id,facility_id,has_active_medicare_b,routing_decision,unknown_risk_score,unknown_flag_count,note_assessment_conflict,multiple_eligible_wounds,envive_narrative_only,length_cm,width_cm,depth_cm,gender,primary_payer_code,active_dx_count,note_count,assessment_count
1,FA-001,101,1,flag_for_review,20,1,0,0,0,2.9,2.8,,Female,MCB,1,1,1
2,FA-002,101,0,reject,25,1,0,0,0,5.9,4.5,,Male,HMO,2,2,1
3,FA-003,101,1,flag_for_review,20,0,0,0,0,8.0,3.5,0.2,Male,MCB,2,1,1
4,FA-004,101,0,reject,15,0,0,0,0,4.3,1.8,0.3,Female,MCD,2,1,1
5,FA-005,101,1,flag_for_review,40,1,0,0,0,3.1,3.4,,Female,MCB,4,1,1
6,FA-006,101,1,flag_for_review,40,1,0,0,0,2.7,1.5,,Female,MCB,2,2,1
7,FA-007,101,0,reject,15,0,0,0,0,2.8,3.8,1.4,Female,MCD,4,2,1
8,FA-008,101,0,reject,40,1,0,0,0,6.0,1.7,,Female,HMO,2,1,1
9,FA-009,101,1,flag_for_review,15,0,0,0,0,5.3,4.5,0.8,Male,MCB,2,2,1
10,FA-010,101,0,reject,40,1,0,0,0,3.7,2.0,,Male,HMO,3,2,1
11,FA-011,101,1,flag_for_review,20,1,0,0,0,1.2,1.5,,Female,MCB,2,1,1
12,FA-012,101,1,auto_accept,10,0,0,0,0,1.2,0.7,0.2,Male,MCB,2,2,1
13,FA-013,101,1,flag_for_review,15,0,0,0,0,1.4,1.2,0.8,Male,MCB,2,1,1
14,FA-014,101,0,reject,35,2,0,1,0,1.0,1.9,,Female,HMO,3,2,1
15,FA-015,101,0,reject,15,0,0,0,0,6.4,4.5,0.6,Male,HMO,2,2,1
16,FA-016,101,1,flag_for_review,30,0,0,0,0,,,,Female,MCB,2,1,1
17,FA-017,101,1,flag_for_review,15,0,0,0,0,6.3,5.7,0.2,Male,MCB,2,2,1
18,FA-018,101,0,reject,20,0,0,0,0,1.1,2.9,0.3,Male,MCA,4,1,1
19,FA-019,101,0,reject,20,1,0,1,0,1.2,1.5,,Male,HMO,2,2,1
20,FA-020,101,1,flag_for_review,40,1,0,0,0,3.2,2.3,,Male,MCB,3,1,1
21,FA-021,101,1,flag_for_review,15,0,0,0,0,2.6,1.3,0.5,Female,MCB,2,1,1
22,FA-022,101,0,reject,40,1,0,0,0,2.5,3.2,,Male,MCA,3,2,1
23,FA-023,101,1,flag_for_review,20,0,0,0,0,4.1,3.6,0.7,Female,MCB,3,2,1
24,FA-024,101,1,flag_for_review,20,1,0,0,0,6.8,4.4,,Male,MCB,2,1,1
25,FA-025,101,1,flag_for_review,25,0,0,0,0,,,,Female,MCB,2,1,1
26,FA-026,101,1,flag_for_review,15,0,0,0,0,3.8,2.2,1.8,Male,MCB,3,1,1
27,FA-027,101,1,auto_accept,10,0,0,0,0,0.8,1.1,0.6,Male,MCB,3,2,1
28,FA-028,101,0,reject,30,1,0,0,0,1.6,3.2,0.3,Female,MCA,2,2,1
29,FA-029,101,0,reject,20,1,0,0,0,4.3,5.4,,Female,MCA,3,2,1
30,FA-030,101,0,reject,5,0,0,0,0,3.9,3.6,0.6,Male,HMO,4,1,1
31,FA-031,101,0,reject,10,0,0,0,0,0.8,1.5,0.6,Male,MCD,1,1,1
32,FA-032,101,0,reject,30,1,0,0,0,4.2,1.3,0.8,Female,HMO,3,2,1
33,FA-033,101,0,reject,15,0,0,0,0,4.7,3.5,1.0,Female,MCA,2,1,1
34,FA-034,101,1,flag_for_review,20,1,0,0,0,3.0,2.2,,Female,MCB,1,1,1
35,FA-035,101,0,reject,25,1,0,0,0,0.8,0.7,0.7,Female,HMO,3,2,1
36,FA-036,101,0,reject,15,0,0,0,0,1.6,2.7,1.2,Male,HMO,4,1,1
37,FA-037,101,0,reject,0,0,0,0,0,,,,Male,HMO,0,0,0
38,FA-038,101,1,flag_for_review,15,0,0,1,0,3.1,1.7,0.3,Female,MCB,3,2,1
39,FA-039,101,1,flag_for_review,15,0,0,0,0,1.4,1.7,0.7,Male,MCB,2,2,1
40,FA-040,101,1,flag_for_review,40,1,0,0,0,3.0,0.9,,Male,MCB,3,2,1
41,FA-041,101,1,flag_for_review,15,0,0,0,0,1.3,0.8,0.8,Female,MCB,3,2,1
42,FA-042,101,0,reject,15,0,0,0,0,5.2,2.4,0.2,Female,HMO,2,1,1
43,FA-043,101,1,flag_for_review,35,1,0,0,0,1.9,0.6,,Female,MCB,2,1,1
44,FA-044,101,1,flag_for_review,35,1,0,0,0,2.8,1.8,,Male,MCB,2,2,1
45,FA-045,101,0,reject,35,1,0,0,0,1.2,0.9,,Female,MCA,4,1,1
46,FA-046,101,0,reject,15,0,0,0,0,2.9,3.1,0.8,Male,HMO,4,1,1
47,FA-047,101,1,flag_for_review,40,1,0,0,0,2.2,1.0,,Female,MCB,3,1,1
48,FA-048,101,1,flag_for_review,20,0,0,1,0,5.4,5.4,0.2,Female,MCB,4,2,1
49,FA-049,101,1,flag_for_review,20,0,0,1,0,3.8,4.1,0.9,Male,MCB,4,2,1
50,FA-050,101,1,flag_for_review,40,1,0,0,0,0.7,2.5,,Male,MCB,2,1,1
51,FA-051,101,1,flag_for_review,25,0,0,0,0,,,,Female,MCB,2,1,1
52,FA-052,101,1,flag_for_review,40,1,0,0,0,2.8,3.2,,Female,MCB,3,1,1
53,FA-053,101,1,flag_for_review,20,1,0,0,0,6.7,4.5,,Male,MCB,3,1,1
54,FA-054,101,0,reject,55,2,0,1,0,1.0,2.1,,Female,MCA,4,2,1
55,FA-055,101,1,auto_accept,25,1,0,0,0,0.5,1.5,0.4,Male,MCB,3,2,1
56,FA-056,101,0,reject,35,0,0,0,0,,,,Male,MCB,0,0,1
57,FA-057,101,1,flag_for_review,35,1,0,0,0,0.8,0.8,,Female,MCB,2,1,1
58,FA-058,101,0,reject,25,0,0,0,0,,,,Male,MCD,3,1,1
59,FA-059,101,1,flag_for_review,50,2,0,1,0,1.6,1.1,,Female,MCB,2,2,1
60,FA-060,101,1,flag_for_review,20,0,0,0,0,3.5,5.4,0.7,Female,MCB,2,2,1
61,FA-061,101,0,reject,15,0,0,0,0,2.8,1.7,0.6,Male,HMO,4,1,1
62,FA-062,101,0,reject,15,0,0,0,0,4.1,0.9,0.3,Male,MCD,4,1,1
63,FA-063,101,0,reject,30,0,0,0,0,,,,Female,MCA,4,1,1
64,FA-064,101,0,reject,15,0,0,0,0,4.0,1.8,0.8,Male,MCD,2,2,1
65,FA-065,101,0,reject,25,1,0,0,0,3.1,4.8,0.2,Female,HMO,2,2,1
66,FA-066,101,1,flag_for_review,25,0,0,0,0,,,,Male,MCB,3,1,1
67,FA-067,101,1,flag_for_review,30,1,0,0,0,4.8,1.0,1.9,Female,MCB,2,2,1
68,FA-068,101,1,auto_accept,5,0,0,0,0,4.0,2.2,0.3,Male,MCB,2,1,1
69,FA-069,101,1,auto_accept,10,0,0,0,0,3.4,2.1,1.7,Male,MCB,3,1,1
70,FA-070,101,1,auto_accept,25,1,0,0,0,2.6,3.5,0.2,Female,MCB,4,2,1
71,FA-071,101,1,flag_for_review,30,0,0,1,0,,,,Female,MCB,3,2,1
72,FA-072,101,1,flag_for_review,15,0,0,1,0,2.5,4.5,0.2,Female,MCB,2,2,1
73,FA-073,101,0,reject,5,0,0,0,0,9.3,7.4,2.2,Female,MCA,4,2,1
74,FA-074,101,0,reject,40,1,0,0,0,1.1,1.4,,Female,MCA,2,2,1
75,FA-075,101,0,reject,15,0,0,0,0,2.1,1.3,0.5,Male,MCA,2,2,1
76,FA-076,101,0,reject,35,1,0,0,0,0.5,1.1,,Male,MCD,4,1,1
77,FA-077,101,1,flag_for_review,15,0,0,0,0,4.3,4.7,1.2,Female,MCB,2,1,1
78,FA-078,101,0,reject,15,0,0,0,0,1.7,3.4,0.6,Female,HMO,2,2,1
79,FA-079,101,1,flag_for_review,15,0,0,0,0,2.1,3.3,1.1,Female,MCB,4,2,1
80,FA-080,101,1,flag_for_review,15,0,0,0,0,2.1,2.0,1.3,Female,MCB,3,1,1
81,FA-081,101,0,reject,35,1,0,0,0,1.7,1.4,,Female,HMO,3,1,1
82,FA-082,101,0,reject,15,0,0,0,0,4.0,2.2,1.5,Male,MCA,4,2,1
83,FA-083,101,1,flag_for_review,15,0,0,0,0,1.6,1.7,0.9,Male,MCB,2,1,1
84,FA-084,101,0,reject,30,1,0,0,0,4.0,2.6,1.3,Female,HMO,4,2,1
85,FA-085,101,0,reject,10,0,0,0,0,2.0,3.8,0.0,Male,HMO,4,1,1
86,FA-086,101,0,reject,25,0,0,0,0,,,,Female,MCA,3,1,1
87,FA-087,101,0,reject,15,0,0,0,0,2.4,3.4,1.5,Female,MCD,2,1,1
88,FA-088,101,1,flag_for_review,15,0,0,0,0,1.9,3.2,1.2,Female,MCB,3,2,1
89,FA-089,101,1,flag_for_review,40,1,0,0,0,3.6,3.1,,Female,MCB,3,1,1
90,FA-090,101,0,reject,40,1,0,0,0,1.1,2.0,,Female,MCD,4,1,1
91,FA-091,101,1,flag_for_review,30,1,0,0,0,3.6,4.5,0.5,Female,MCB,3,2,1
92,FA-092,101,0,reject,20,0,0,0,0,5.4,1.7,0.5,Female,HMO,4,1,1
93,FA-093,101,0,reject,40,1,0,0,0,4.0,3.0,,Female,MCD,2,1,1
94,FA-094,101,0,reject,30,0,0,0,0,,,,Female,MCA,2,1,1
95,FA-095,101,0,reject,15,0,0,1,0,5.6,5.0,0.7,Female,MCA,3,2,1
96,FA-096,101,1,flag_for_review,30,0,0,0,0,,,,Female,MCB,2,1,1
97,FA-097,101,0,reject,15,0,0,0,0,2.4,2.6,0.4,Female,MCD,2,2,1
98,FA-098,101,1,flag_for_review,30,1,0,0,0,4.7,2.1,2.0,Male,MCB,2,2,1
99,FA-099,101,1,flag_for_review,40,1,0,0,0,3.8,1.0,,Female,MCB,3,1,1
100,FA-100,101,1,flag_for_review,40,1,0,1,0,1.3,1.1,,Male,MCB,2,2,1
101,FA-101,101,0,reject,15,0,0,1,0,9.2,4.6,2.7,Male,HMO,2,2,1
102,FA-102,101,0,reject,35,1,0,0,0,1.9,2.0,,Male,MCA,4,2,1
103,FA-103,101,0,reject,25,0,0,1,0,,,,Male,MCA,3,2,1
104,FA-104,101,0,reject,20,1,0,1,0,1.4,1.4,,Female,HMO,2,2,1
105,FA-105,101,0,reject,20,0,0,0,0,5.7,1.6,0.5,Female,HMO,1,1,1
106,FA-106,101,0,reject,30,1,0,1,0,0.6,0.9,0.2,Male,MCD,3,2,1
107,FA-107,101,0,reject,30,0,0,0,0,,,,Female,HMO,3,1,1
108,FA-108,101,1,flag_for_review,10,0,0,1,0,1.6,4.0,0.1,Female,MCB,3,2,1
109,FA-109,101,1,flag_for_review,40,1,0,0,0,1.1,1.8,,Female,MCB,3,1,1
110,FA-110,101,1,flag_for_review,20,0,0,0,0,2.4,4.5,0.1,Female,MCB,4,2,1
111,FA-111,101,1,flag_for_review,40,1,0,0,0,1.9,1.9,,Female,MCB,3,1,1
112,FA-112,101,0,reject,15,0,0,1,0,4.4,3.4,1.4,Female,HMO,4,2,1
113,FA-113,101,0,reject,35,1,0,0,0,3.9,3.1,,Male,MCA,4,1,1
114,FA-114,101,0,reject,15,0,0,1,0,4.9,2.2,1.2,Female,HMO,2,2,1
115,FA-115,101,0,reject,15,0,0,0,0,2.7,1.1,0.7,Male,MCD,3,1,1
116,FA-116,101,0,reject,40,1,0,0,0,1.4,3.1,,Male,MCA,3,2,1
117,FA-117,101,0,reject,25,0,0,0,0,,,,Female,HMO,4,1,1
118,FA-118,101,1,flag_for_review,40,1,0,0,0,6.1,2.4,,Male,MCB,3,2,1
119,FA-119,101,1,flag_for_review,15,0,0,1,0,2.4,0.5,0.8,Male,MCB,3,2,1
120,FA-120,101,1,flag_for_review,20,0,0,0,0,3.7,3.1,1.1,Male,MCB,4,1,1
121,FB-001,102,0,reject,15,0,0,0,0,4.4,4.2,1.0,Male,MCD,3,1,1
122,FB-002,102,0,reject,15,0,0,0,0,2.5,1.9,0.9,Female,MCA,2,2,1
123,FB-003,102,0,reject,40,1,0,0,0,7.1,5.5,,Female,HMO,2,1,1
124,FB-004,102,0,reject,25,0,0,0,0,,,,Male,HMO,3,1,1
125,FB-005,102,0,reject,40,1,0,0,0,3.3,2.3,,Female,HMO,4,1,1
126,FB-006,102,1,flag_for_review,40,1,0,0,0,3.2,3.7,,Female,MCB,4,2,1
127,FB-007,102,0,reject,5,0,0,0,0,3.5,3.4,0.3,Female,HMO,2,1,1
128,FB-008,102,1,flag_for_review,15,0,0,0,0,4.3,2.4,1.2,Male,MCB,4,1,1
129,FB-009,102,0,reject,20,1,0,0,0,1.3,2.7,,Male,HMO,2,2,1
130,FB-010,102,0,reject,15,0,0,0,0,2.6,0.7,0.7,Female,MCA,4,1,1
131,FB-011,102,0,reject,15,0,0,0,0,5.4,2.4,0.6,Female,MCA,2,1,1
132,FB-012,102,1,auto_accept,25,1,0,0,0,1.3,2.3,0.0,Female,MCB,3,2,1
133,FB-013,102,0,reject,20,0,0,1,0,5.2,3.6,3.2,Male,MCA,2,2,1
134,FB-014,102,1,auto_accept,10,0,0,0,0,2.0,0.2,0.6,Female,MCB,2,2,1
135,FB-015,102,1,flag_for_review,40,1,0,0,0,0.6,1.2,,Female,MCB,3,2,1
136,FB-016,102,1,flag_for_review,30,0,0,0,0,,,,Female,MCB,3,1,1
137,FB-017,102,0,reject,30,1,0,0,0,1.1,3.8,1.2,Female,HMO,2,2,1
138,FB-018,102,0,reject,25,1,0,0,0,5.4,2.8,,Female,MCA,3,2,1
139,FB-019,102,1,flag_for_review,40,1,0,0,0,4.8,3.5,,Female,MCB,4,2,1
140,FB-020,102,1,flag_for_review,30,0,0,1,0,,,,Male,MCB,3,2,1
141,FB-021,102,1,flag_for_review,20,1,0,0,0,2.9,4.4,,Male,MCB,4,1,1
142,FB-022,102,0,reject,20,0,0,0,0,7.8,1.9,0.6,Female,MCD,3,1,1
143,FB-023,102,0,reject,15,0,0,0,0,7.3,5.0,0.8,Male,HMO,3,1,1
144,FB-024,102,0,reject,15,0,0,0,0,1.9,3.8,0.2,Male,MCA,2,1,1
145,FB-025,102,0,reject,35,2,0,1,0,1.9,1.9,,Female,MCA,4,2,1
146,FB-026,102,1,auto_accept,5,0,0,0,0,2.7,3.2,0.2,Male,MCB,2,2,1
147,FB-027,102,1,flag_for_review,40,1,0,0,0,3.7,2.6,,Male,MCB,4,1,1
148,FB-028,102,1,flag_for_review,30,1,0,0,0,1.1,2.2,0.6,Female,MCB,2,2,1
149,FB-029,102,1,flag_for_review,15,0,0,0,0,5.8,2.6,0.1,Female,MCB,2,2,1
150,FB-030,102,0,reject,25,0,0,0,0,,,,Female,MCA,2,1,1
151,FB-031,102,1,flag_for_review,35,1,0,1,0,1.6,0.5,,Female,MCB,2,2,1
152,FB-032,102,1,flag_for_review,40,1,0,1,0,4.9,1.6,,Male,MCB,2,2,1
153,FB-033,102,1,flag_for_review,40,1,0,0,0,4.3,3.2,,Male,MCB,2,2,1
154,FB-034,102,1,flag_for_review,25,0,0,0,0,,,,Female,MCB,3,1,1
155,FB-035,102,1,flag_for_review,30,1,0,0,0,3.4,1.1,0.7,Male,MCB,2,2,1
156,FB-036,102,0,reject,30,1,0,1,0,1.8,0.8,0.2,Female,HMO,4,2,1
157,FB-037,102,1,flag_for_review,20,1,0,0,0,2.5,2.4,,Male,MCB,3,2,1
158,FB-038,102,1,flag_for_review,30,1,0,0,0,2.8,3.8,0.7,Male,MCB,2,2,1
159,FB-039,102,0,reject,15,0,0,1,0,6.2,3.0,0.4,Female,MCA,3,2,1
160,FB-040,102,0,reject,40,1,0,0,0,5.6,5.4,,Female,HMO,3,2,1
161,FB-041,102,0,reject,5,0,0,0,0,1.3,1.6,0.4,Female,MCA,2,2,1
162,FB-042,102,0,reject,5,0,0,0,0,1.2,1.8,0.3,Female,MCD,4,1,1
163,FB-043,102,1,flag_for_review,30,1,0,0,0,3.7,2.1,1.3,Female,MCB,4,2,1
164,FB-044,102,1,flag_for_review,55,2,0,1,0,6.9,1.7,,Male,MCB,3,2,1
165,FB-045,102,0,reject,15,0,0,0,0,3.4,2.4,1.4,Male,HMO,3,2,1
166,FB-046,102,0,reject,20,0,0,0,0,4.7,0.9,0.9,Female,MCA,3,1,1
167,FB-047,102,0,reject,15,0,0,0,0,4.7,3.7,2.9,Male,MCD,2,2,1
168,FB-048,102,0,reject,10,0,0,0,0,1.1,2.0,0.2,Male,MCA,3,1,1
169,FB-049,102,0,reject,55,2,0,1,0,2.3,3.5,,Male,HMO,3,2,1
170,FB-050,102,0,reject,5,0,0,0,0,3.3,2.5,0.3,Male,MCA,2,1,1
171,FB-051,102,0,reject,25,1,0,1,0,5.2,2.2,,Female,MCA,1,2,1
172,FB-052,102,0,reject,20,0,0,0,0,4.6,2.4,1.4,Male,MCD,2,1,1
173,FB-053,102,0,reject,30,0,0,0,0,,,,Male,HMO,2,1,1
174,FB-054,102,0,reject,10,0,0,0,0,5.4,4.4,0.2,Female,MCD,2,1,1
175,FB-055,102,0,reject,55,2,0,1,0,3.1,1.5,,Female,MCA,2,2,1
176,FB-056,102,1,auto_accept,10,0,0,0,0,1.1,3.5,0.2,Male,MCB,3,2,1
177,FB-057,102,0,reject,15,0,0,1,0,1.8,1.1,0.9,Male,MCD,2,2,1
178,FB-058,102,1,flag_for_review,40,1,0,0,0,2.2,1.8,,Male,MCB,2,2,1
179,FB-059,102,0,reject,10,0,0,0,0,3.8,2.7,1.0,Female,MCA,3,2,1
180,FB-060,102,0,reject,15,0,0,0,0,1.0,0.9,0.1,Female,MCA,2,1,1
181,FB-061,102,1,flag_for_review,30,0,0,0,0,,,,Male,MCB,3,1,1
182,FB-062,102,0,reject,25,1,0,1,0,5.1,1.6,1.6,Male,MCA,3,2,1
183,FB-063,102,0,reject,30,1,0,0,0,2.9,1.3,0.3,Female,MCD,2,2,1
184,FB-064,102,0,reject,20,1,0,0,0,3.1,2.5,,Male,HMO,4,2,1
185,FB-065,102,1,flag_for_review,15,0,0,0,0,4.8,3.3,1.4,Male,MCB,4,1,1
186,FB-066,102,1,flag_for_review,20,1,0,0,0,2.1,2.2,,Female,MCB,2,2,1
187,FB-067,102,1,flag_for_review,15,0,0,0,0,2.3,1.1,0.4,Female,MCB,3,1,1
188,FB-068,102,0,reject,30,1,0,0,0,4.1,2.7,1.4,Male,HMO,4,2,1
189,FB-069,102,1,flag_for_review,15,0,0,0,0,4.4,2.6,1.4,Female,MCB,3,2,1
190,FB-070,102,0,reject,30,1,0,0,0,1.6,4.6,0.5,Male,MCD,2,2,1
191,FB-071,102,0,reject,20,0,0,0,0,1.5,0.6,0.2,Female,MCD,2,1,1
192,FB-072,102,1,flag_for_review,15,0,0,0,0,2.6,3.2,1.0,Female,MCB,3,1,1
193,FB-073,102,0,reject,15,0,0,0,0,4.9,1.6,0.8,Male,HMO,2,2,1
194,FB-074,102,0,reject,30,0,0,1,0,,,,Female,HMO,4,2,1
195,FB-075,102,0,reject,10,0,0,0,0,5.5,3.3,0.7,Female,HMO,3,2,1
196,FB-076,102,0,reject,5,0,0,0,0,4.6,3.3,1.2,Male,MCD,2,2,1
197,FB-077,102,0,reject,15,0,0,1,0,1.6,2.4,0.5,Female,MCA,2,2,1
198,FB-078,102,1,flag_for_review,15,0,0,0,0,2.0,2.4,0.9,Male,MCB,3,1,1
199,FB-079,102,1,auto_accept,25,1,0,0,0,5.5,4.0,0.2,Female,MCB,4,2,1
200,FB-080,102,1,flag_for_review,25,0,0,0,0,,,,Male,MCB,2,1,1
201,FB-081,102,0,reject,35,1,0,0,0,1.7,0.5,,Male,MCD,3,1,1
202,FB-082,102,0,reject,20,0,0,1,0,5.7,1.2,0.3,Female,MCA,4,2,1
203,FB-083,102,1,flag_for_review,20,1,0,0,0,5.2,2.6,,Female,MCB,2,1,1
204,FB-084,102,1,flag_for_review,55,2,0,1,0,4.5,2.0,,Male,MCB,3,2,1
205,FB-085,102,1,flag_for_review,20,0,0,0,0,8.2,6.3,1.7,Female,MCB,4,1,1
206,FB-086,102,0,reject,25,1,0,0,0,3.6,1.7,,Male,MCD,3,2,1
207,FB-087,102,1,flag_for_review,55,2,0,1,0,3.9,3.9,,Female,MCB,3,2,1
208,FB-088,102,1,flag_for_review,30,0,0,0,0,,,,Male,MCB,4,1,1
209,FB-089,102,1,flag_for_review,15,0,0,0,0,2.2,1.5,0.4,Female,MCB,3,1,1
210,FB-090,102,0,reject,30,0,0,0,0,,,,Male,HMO,2,1,1
211,FC-001,103,1,flag_for_review,20,0,0,0,0,4.3,4.4,1.3,Female,MCB,4,1,1
212,FC-002,103,1,flag_for_review,15,0,0,0,0,3.6,2.3,0.7,Male,MCB,3,2,1
213,FC-003,103,1,flag_for_review,20,1,0,0,0,6.2,4.5,,Female,MCB,2,2,1
214,FC-004,103,0,reject,10,0,0,0,0,5.2,3.4,0.1,Male,MCA,2,2,1
215,FC-005,103,1,flag_for_review,30,0,0,0,0,,,,Female,MCB,4,1,1
216,FC-006,103,1,flag_for_review,30,0,0,0,0,,,,Male,MCB,4,1,1
217,FC-007,103,0,reject,15,0,0,1,0,2.7,2.4,0.8,Male,MCD,2,2,1
218,FC-008,103,1,flag_for_review,15,0,0,0,0,1.6,3.3,0.7,Female,MCB,3,2,1
219,FC-009,103,1,flag_for_review,40,1,0,0,0,3.3,1.2,,Female,MCB,4,2,1
220,FC-010,103,0,reject,40,1,0,0,0,1.5,3.7,,Female,MCD,2,1,1
221,FC-011,103,0,reject,30,1,0,0,0,4.7,1.4,0.4,Male,MCA,3,2,1
222,FC-012,103,1,flag_for_review,25,0,0,0,0,,,,Male,MCB,3,1,1
223,FC-013,103,0,reject,15,0,0,0,0,1.7,3.9,0.6,Male,MCA,3,2,1
224,FC-014,103,1,flag_for_review,25,0,0,0,0,,,,Male,MCB,2,1,1
225,FC-015,103,1,flag_for_review,35,1,0,1,0,1.8,1.8,0.6,Male,MCB,2,2,1
226,FC-016,103,1,auto_accept,5,0,0,0,0,5.8,2.2,0.9,Male,MCB,4,2,1
227,FC-017,103,1,flag_for_review,15,0,0,0,0,2.7,0.9,0.8,Female,MCB,3,2,1
228,FC-018,103,0,reject,20,0,0,0,0,2.2,1.5,0.7,Male,MCD,2,1,1
229,FC-019,103,0,reject,20,0,0,0,0,1.9,2.2,0.5,Male,MCA,4,1,1
230,FC-020,103,0,reject,15,0,0,0,0,3.0,1.8,0.2,Female,MCD,4,2,1
231,FC-021,103,0,reject,40,1,0,0,0,3.7,1.7,,Male,HMO,2,1,1
232,FC-022,103,0,reject,20,0,0,0,0,3.9,3.8,0.7,Male,HMO,4,2,1
233,FC-023,103,0,reject,25,1,0,0,0,3.4,1.6,,Male,MCA,4,2,1
234,FC-024,103,0,reject,15,0,0,1,0,0.5,0.6,0.5,Male,HMO,2,2,1
235,FC-025,103,0,reject,15,0,0,0,0,3.3,2.9,1.5,Female,HMO,2,2,1
236,FC-026,103,1,flag_for_review,35,1,0,0,0,0.8,0.9,,Male,MCB,2,1,1
237,FC-027,103,1,flag_for_review,30,0,0,1,0,,,,Female,MCB,2,2,1
238,FC-028,103,1,flag_for_review,40,1,0,0,0,3.6,1.9,,Male,MCB,3,2,1
239,FC-029,103,1,flag_for_review,30,1,0,0,0,2.4,4.0,1.8,Female,MCB,2,2,1
240,FC-030,103,0,reject,15,0,0,0,0,3.0,2.8,1.1,Female,MCA,4,2,1
241,FC-031,103,0,reject,15,0,0,1,0,2.3,2.1,0.8,Male,HMO,2,2,1
242,FC-032,103,1,flag_for_review,15,0,0,1,0,3.1,2.4,0.2,Female,MCB,3,2,1
243,FC-033,103,0,reject,30,1,0,0,0,3.0,2.8,0.5,Female,HMO,1,2,1
244,FC-034,103,0,reject,10,0,0,0,0,0.5,1.2,0.2,Male,MCA,3,1,1
245,FC-035,103,0,reject,20,0,0,0,0,4.8,3.4,1.0,Male,HMO,3,2,1
246,FC-036,103,1,flag_for_review,25,1,0,0,0,5.9,2.0,,Male,MCB,4,2,1
247,FC-037,103,0,reject,30,0,0,1,0,,,,Male,HMO,3,2,1
248,FC-038,103,1,auto_accept,10,0,0,0,0,4.3,4.8,1.2,Female,MCB,4,2,1
249,FC-039,103,0,reject,55,2,0,1,0,2.8,4.6,,Female,HMO,2,2,1
250,FC-040,103,0,reject,15,0,0,1,0,3.3,4.7,0.8,Female,MCD,4,2,1
251,FC-041,103,1,flag_for_review,20,1,0,1,0,9.1,5.2,3.3,Male,MCB,4,2,1
252,FC-042,103,1,flag_for_review,25,0,0,0,0,,,,Female,MCB,3,1,1
253,FC-043,103,0,reject,15,0,0,0,0,4.0,4.4,0.2,Male,MCA,3,2,1
254,FC-044,103,0,reject,15,0,0,0,0,2.2,2.6,0.6,Female,HMO,2,2,1
255,FC-045,103,1,flag_for_review,40,1,0,0,0,1.9,2.8,,Male,MCB,2,2,1
256,FC-046,103,0,reject,40,1,0,0,0,1.2,2.0,,Male,MCA,2,1,1
257,FC-047,103,0,reject,35,1,0,0,0,3.7,2.2,,Male,MCA,3,1,1
258,FC-048,103,1,flag_for_review,15,0,0,0,0,2.8,3.1,0.7,Male,MCB,4,1,1
259,FC-049,103,0,reject,25,1,0,0,0,1.6,1.0,0.7,Female,HMO,2,2,1
260,FC-050,103,0,reject,20,1,0,0,0,5.3,4.5,,Female,MCA,4,2,1
261,FC-051,103,1,flag_for_review,15,0,0,0,0,6.3,3.9,0.7,Female,MCB,2,2,1
262,FC-052,103,0,reject,15,0,0,0,0,2.2,5.4,0.6,Female,MCD,1,2,1
263,FC-053,103,0,reject,30,0,0,1,0,,,,Male,MCA,4,2,1
264,FC-054,103,0,reject,40,1,0,0,0,6.2,5.7,,Male,MCD,2,2,1
265,FC-055,103,0,reject,25,1,0,0,0,2.8,1.6,0.2,Male,HMO,2,2,1
266,FC-056,103,1,flag_for_review,35,1,0,0,0,1.1,1.1,,Female,MCB,3,1,1
267,FC-057,103,1,flag_for_review,25,0,0,0,0,,,,Female,MCB,3,1,1
268,FC-058,103,1,flag_for_review,25,1,0,0,0,2.1,2.8,,Female,MCB,3,2,1
269,FC-059,103,0,reject,15,0,0,0,0,1.7,2.0,0.4,Male,MCA,2,2,1
270,FC-060,103,1,flag_for_review,35,2,0,1,0,4.6,4.1,,Female,MCB,4,2,1
271,FC-061,103,0,reject,20,1,0,0,0,6.4,4.7,,Male,MCD,4,2,1
272,FC-062,103,1,flag_for_review,25,0,0,0,0,,,,Female,MCB,2,1,1
273,FC-063,103,0,reject,40,1,0,0,0,2.5,4.1,,Male,MCA,3,1,1
274,FC-064,103,0,reject,20,1,0,0,0,8.4,7.8,,Female,HMO,2,2,1
275,FC-065,103,1,flag_for_review,20,0,0,0,0,1.1,1.4,0.7,Female,MCB,4,1,1
276,FC-066,103,1,flag_for_review,35,1,0,0,0,1.3,1.0,,Male,MCB,3,2,1
277,FC-067,103,1,flag_for_review,15,0,0,0,0,7.0,2.1,0.9,Male,MCB,3,1,1
278,FC-068,103,0,reject,10,0,0,0,0,4.6,3.5,0.1,Female,MCD,3,2,1
279,FC-069,103,0,reject,20,0,0,0,0,2.2,1.2,0.2,Male,HMO,3,1,1
280,FC-070,103,1,flag_for_review,30,0,0,0,0,,,,Male,MCB,2,1,1
281,FC-071,103,1,auto_accept,25,1,0,0,0,1.8,1.2,0.0,Female,MCB,3,2,1
282,FC-072,103,1,flag_for_review,15,0,0,0,0,2.2,4.8,1.0,Male,MCB,3,2,1
283,FC-073,103,1,flag_for_review,20,1,0,1,0,7.8,4.0,3.7,Male,MCB,4,2,1
284,FC-074,103,1,flag_for_review,15,0,0,1,0,6.5,5.0,0.3,Female,MCB,2,2,1
285,FC-075,103,0,reject,20,0,0,0,0,0.5,0.5,0.2,Female,MCA,3,1,1
286,FC-076,103,0,reject,25,0,0,0,0,,,,Male,HMO,3,1,1
287,FC-077,103,1,flag_for_review,20,0,0,1,0,3.5,3.9,0.8,Male,MCB,3,2,1
288,FC-078,103,1,flag_for_review,15,0,0,1,0,5.5,1.8,0.5,Male,MCB,2,2,1
289,FC-079,103,0,reject,15,0,0,0,0,1.2,1.6,0.6,Male,MCA,3,2,1
290,FC-080,103,1,flag_for_review,5,0,0,1,0,6.0,5.0,0.5,Female,MCB,2,2,1
291,FC-081,103,1,flag_for_review,15,0,0,1,0,4.3,3.2,1.2,Female,MCB,3,2,1
292,FC-082,103,1,flag_for_review,15,0,0,0,0,5.5,2.0,1.0,Male,MCB,2,1,1
293,FC-083,103,0,reject,15,0,0,0,0,5.8,3.8,3.1,Male,HMO,3,1,1
294,FC-084,103,0,reject,30,1,0,0,0,0.7,2.0,0.7,Male,HMO,2,2,1
295,FC-085,103,1,flag_for_review,35,1,0,0,0,1.4,0.7,,Female,MCB,3,2,1
296,FC-086,103,0,reject,15,0,0,0,0,3.0,3.8,0.4,Male,MCD,2,1,1
297,FC-087,103,1,flag_for_review,15,0,0,0,0,3.9,2.1,0.3,Male,MCB,4,1,1
298,FC-088,103,0,reject,20,1,0,1,0,3.0,3.2,0.3,Male,MCA,4,2,1
299,FC-089,103,1,auto_accept,5,0,0,0,0,3.8,3.2,0.3,Female,MCB,2,1,1
300,FC-090,103,0,reject,15,0,0,0,0,1.0,2.9,1.3,Female,HMO,3,2,1
"""

Path("features.csv").write_text(FEATURES_CSV)

"""
ABI Wound Care — Decision Tree Training Script (Colab-ready)

Usage in Google Colab:
  1. Upload `features.csv` (export via: python backend/cli.py export-features)
  2. Run all cells /: python ml/train_model.py --features features.csv --out decision_tree.joblib
  3. Download decision_tree.joblib and place in ml/models/
  4. Restart the API to load model insights

This script trains a shallow decision tree on rule-derived labels.
The tree is advisory only — it does not override hard eligibility rules.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

FEATURE_COLUMNS = [
    "facility_id",
    "has_active_medicare_b",
    "unknown_risk_score",
    "unknown_flag_count",
    "note_assessment_conflict",
    "multiple_eligible_wounds",
    "envive_narrative_only",
    "length_cm",
    "width_cm",
    "depth_cm",
    "active_dx_count",
    "note_count",
    "assessment_count",
]

TARGET_COLUMN = "target_auto_accept"


def build_target(df: pd.DataFrame) -> pd.Series:
    """Binary target: 1 if rules engine said auto_accept, else 0."""
    return (df["routing_decision"] == "auto_accept").astype(int)


def train(features_path: str, out_path: str) -> dict:
    df = pd.read_csv(features_path)
    if df.empty:
        raise ValueError("Feature CSV is empty. Run sync → extract → decide → export-features first.")

    df[TARGET_COLUMN] = build_target(df)

    X = df[FEATURE_COLUMNS].copy()
    y = df[TARGET_COLUMN]

    numeric_features = [
        "facility_id",
        "has_active_medicare_b",
        "unknown_risk_score",
        "unknown_flag_count",
        "note_assessment_conflict",
        "multiple_eligible_wounds",
        "envive_narrative_only",
        "length_cm",
        "width_cm",
        "depth_cm",
        "active_dx_count",
        "note_count",
        "assessment_count",
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                ]),
                numeric_features,
            )
        ]
    )

    tree = Pipeline([
        ("prep", preprocessor),
        (
            "clf",
            DecisionTreeClassifier(
                max_depth=4,
                min_samples_leaf=15,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None)

    cv_scores = cross_val_score(tree, X, y, cv=min(5, len(df)), scoring="accuracy")
    tree.fit(X_train, y_train)
    y_pred = tree.predict(X_test)

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    # Logistic baseline
    log_reg = Pipeline([
        ("prep", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])
    log_reg.fit(X_train, y_train)
    log_acc = log_reg.score(X_test, y_test)

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    artifact = {
        "pipeline": tree,
        "feature_columns": FEATURE_COLUMNS,
        "cv_accuracy_mean": float(cv_scores.mean()),
        "cv_accuracy_std": float(cv_scores.std()),
        "test_accuracy": float(tree.score(X_test, y_test)),
        "logistic_baseline_accuracy": float(log_acc),
        "classification_report": report,
    }
    joblib.dump(artifact, out_path)

    # Feature importances from fitted tree
    clf: DecisionTreeClassifier = tree.named_steps["clf"]
    importances = dict(zip(FEATURE_COLUMNS, clf.feature_importances_.tolist()))

    summary = {
        "model_path": out_path,
        "rows": len(df),
        "cv_accuracy": artifact["cv_accuracy_mean"],
        "test_accuracy": artifact["test_accuracy"],
        "logistic_baseline": artifact["logistic_baseline_accuracy"],
        "feature_importances": importances,
    }
    summary_path = Path(out_path).with_suffix(".summary.json")
    summary_path.write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))
    return summary


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--features", default="ml/exports/features.csv")
    parser.add_argument("--out", default="ml/models/decision_tree.joblib")
    args = parser.parse_args()
    train(args.features, args.out)




summary = train("features.csv", "decision_tree.joblib")

from google.colab import files
files.download("decision_tree.joblib")
files.download("decision_tree.summary.json")
print("Done — save decision_tree.joblib to Pulse/ml/models/")

